# 4. Data Preprocessing and Feature Engineering

This notebook prepares the cleaned hospital encounter dataset for predictive modeling of 30-day readmission.

The preprocessing workflow includes feature selection, removal of redundant variables, preparation of numerical and categorical predictors, feature engineering, train-test splitting, and construction of preprocessing pipelines. Particular attention is given to preventing data leakage by ensuring that transformations required for model fitting are learned from the training data only.

In [18]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Modeling utilities
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit

## 4.1 Load the Cleaned Dataset

The cleaned and mapped dataset produced during the preceding data preparation stage is loaded as the starting point for model preprocessing. A separate modeling dataframe is created so that subsequent transformations do not modify the source dataset.

In [2]:
# Load cleaned dataset
df = pd.read_csv("../data/processed/diabetic_data_cleaned.csv")

# Create modeling copy
df_model = df.copy()

print("Dataset shape:", df_model.shape)
df_model.head()

Dataset shape: (101763, 46)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,tolazamide,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [3]:
# Inspect available columns
print(f"Number of columns: {df_model.shape[1]}\n")

for col in df_model.columns:
    print(col)

Number of columns: 46

encounter_id
patient_nbr
race
gender
age
admission_type_id
discharge_disposition_id
admission_source_id
time_in_hospital
medical_specialty
num_lab_procedures
num_procedures
num_medications
number_outpatient
number_emergency
number_inpatient
diag_1
diag_2
diag_3
number_diagnoses
max_glu_serum
A1Cresult
metformin
repaglinide
nateglinide
chlorpropamide
glimepiride
acetohexamide
glipizide
glyburide
tolbutamide
pioglitazone
rosiglitazone
acarbose
miglitol
troglitazone
tolazamide
insulin
glyburide-metformin
glipizide-metformin
glimepiride-pioglitazone
metformin-rosiglitazone
metformin-pioglitazone
change
diabetesMed
readmitted


## 4.2 Define the Target Variable

The original readmission variable contains three categories: readmission within 30 days, readmission after 30 days, and no recorded readmission.

For this analysis, the prediction task is formulated as binary classification. Encounters followed by readmission within 30 days are assigned to the positive class (`1`), while encounters with readmission after 30 days or no readmission are assigned to the negative class (`0`).

In [4]:
# Create binary 30-day readmission target
df_model["readmitted_30"] = (
    df_model["readmitted"] == "<30"
).astype(int)

# Check target distribution
target_summary = pd.DataFrame({
    "Count": df_model["readmitted_30"].value_counts(),
    "Percentage": (
        df_model["readmitted_30"]
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
    )
})

target_summary

,Count,Percentage
readmitted_30,,
0,90406,88.8
1,11357,11.2


## 4.3 Remove the Original Readmission Variable

Following creation of the binary target, the original `readmitted` variable is removed from the modeling dataset. Retaining this variable would introduce direct target leakage because the `<30` category is used to define the positive class.

In [5]:
# Remove original target variable
df_model.drop(columns="readmitted", inplace=True)

print("Dataset shape:", df_model.shape)
print("Original readmitted column present:",
      "readmitted" in df_model.columns)

print("Binary target present:",
      "readmitted_30" in df_model.columns)

Dataset shape: (101763, 46)
Original readmitted column present: False
Binary target present: True


## 4.4 Feature Selection

Feature selection is performed before model preprocessing to remove variables that are unsuitable for prediction while retaining potentially informative patient, encounter, clinical, and treatment characteristics.

Variables are evaluated based on their role in the dataset rather than being removed solely because they contain many categories or have weak individual associations with the target.

### 4.4.1 Remove Encounter Identifier

`encounter_id` uniquely identifies each hospital encounter and does not represent a meaningful patient or clinical characteristic. It is therefore excluded from the predictor set.

`patient_nbr` is temporarily retained because it identifies encounters belonging to the same patient and will be used when constructing patient-aware training and test partitions. It will not be used as a model predictor.

In [6]:
# Remove encounter identifier
df_model.drop(columns="encounter_id", inplace=True)

print("Dataset shape:", df_model.shape)

Dataset shape: (101763, 45)


### 4.4.2 Inspect Categorical Feature Cardinality

Before applying categorical encoding, the number of unique values within each categorical feature is examined. This helps distinguish low-cardinality variables that can be encoded directly from higher-cardinality variables that may require additional feature engineering.

Particular attention is given to diagnosis codes and medical specialty because these variables contain many distinct categories. Their cardinality will inform whether categories should be grouped or transformed before model training.

In [7]:
# Select categorical variables
categorical_cols = df_model.select_dtypes(
    include=["object", "string"]
).columns

# Inspect categorical feature cardinality
categorical_summary = pd.DataFrame({
    "Unique Values": df_model[categorical_cols].nunique(),
    "Missing Values": df_model[categorical_cols].isna().sum()
}).sort_values("Unique Values", ascending=False)

categorical_summary

,Unique Values,Missing Values
diag_3,790,0
diag_2,749,0
diag_1,717,0
medical_specialty,73,0
age,10,0
race,6,0
glipizide,4,0
glyburide-metformin,4,0
insulin,4,0
miglitol,4,0


## 4.5 Diagnosis Feature Engineering

The three diagnosis variables (`diag_1`, `diag_2`, and `diag_3`) contain hundreds of distinct ICD-9 diagnosis codes. Direct one-hot encoding of these raw codes would create a large number of sparse features and may reduce model interpretability.

To reduce dimensionality while retaining clinically meaningful information, the raw diagnosis codes are grouped into broader diagnostic categories. Separate grouped features are created for the primary, secondary, and additional diagnoses.

The original diagnosis-code variables will subsequently be removed from the predictor set after the grouped features have been created.

### 4.5.1 Group ICD-9 Diagnosis Codes

The raw diagnosis variables contain hundreds of individual ICD-9 codes, resulting in high-cardinality categorical features. To create more manageable and clinically interpretable predictors, the diagnosis codes are grouped into broader diagnostic categories.

The grouping is applied independently to the primary (`diag_1`), secondary (`diag_2`), and additional (`diag_3`) diagnosis variables. The resulting features preserve broad diagnostic information while substantially reducing the number of categories that will require encoding during model preprocessing.

Unknown diagnosis values are retained as a separate category, while supplementary and less common diagnosis codes are grouped under `Other`.

In [8]:
def categorize_diagnosis(code):
    # Preserve unknown diagnoses
    if pd.isna(code) or str(code).strip().lower() == "unknown":
        return "Unknown"

    code = str(code).strip()

    # Supplementary ICD-9 codes
    if code.startswith(("V", "E")):
        return "Other"

    try:
        code = float(code)
    except ValueError:
        return "Other"

    if 390 <= code <= 459 or code == 785:
        return "Circulatory"
    elif 460 <= code <= 519 or code == 786:
        return "Respiratory"
    elif 520 <= code <= 579 or code == 787:
        return "Digestive"
    elif 250 <= code < 251:
        return "Diabetes"
    elif 800 <= code <= 999:
        return "Injury"
    elif 710 <= code <= 739:
        return "Musculoskeletal"
    elif 580 <= code <= 629 or code == 788:
        return "Genitourinary"
    elif 140 <= code <= 239:
        return "Neoplasms"
    else:
        return "Other"

In [9]:
# Create grouped diagnosis features
for col in ["diag_1", "diag_2", "diag_3"]:
    df_model[f"{col}_group"] = df_model[col].apply(categorize_diagnosis)

# Review the new categories
for col in ["diag_1_group", "diag_2_group", "diag_3_group"]:
    print(f"\n{col}")
    print(df_model[col].value_counts())


diag_1_group
diag_1_group
Circulatory        30436
Other              18172
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6972
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Unknown               21
Name: count, dtype: int64

diag_2_group
diag_2_group
Circulatory        31880
Other              26553
Diabetes           12794
Respiratory        10895
Genitourinary       8376
Digestive           4170
Neoplasms           2547
Injury              2426
Musculoskeletal     1764
Unknown              358
Name: count, dtype: int64

diag_3_group
diag_3_group
Circulatory        30305
Other              29194
Diabetes           17157
Respiratory         7358
Genitourinary       6680
Digestive           3930
Injury              1945
Musculoskeletal     1915
Neoplasms           1856
Unknown             1423
Name: count, dtype: int64


### 4.5.2 Remove Raw Diagnosis Codes

Following creation and validation of the grouped diagnosis features, the original `diag_1`, `diag_2`, and `diag_3` variables are removed from the modeling dataset.

The grouped diagnosis features are retained as lower-dimensional representations of the underlying diagnostic information. This reduces the number of categorical levels that will require encoding while preserving broad clinical information about each encounter.

In [10]:
# Remove raw high-cardinality diagnosis codes
raw_diagnosis_cols = ["diag_1", "diag_2", "diag_3"]

df_model.drop(columns=raw_diagnosis_cols, inplace=True)

print("Dataset shape:", df_model.shape)

print(
    "Raw diagnosis columns present:",
    any(col in df_model.columns for col in raw_diagnosis_cols)
)

print(
    "Grouped diagnosis columns:",
    [col for col in df_model.columns if col.endswith("_group")]
)

Dataset shape: (101763, 45)
Raw diagnosis columns present: False
Grouped diagnosis columns: ['diag_1_group', 'diag_2_group', 'diag_3_group']


## 4.6 Medical Specialty Feature Engineering

`medical_specialty` contains substantially more categories than most of the remaining categorical predictors. Although specialty may provide useful information about the clinical context of an encounter, directly encoding all specialties can create numerous sparse features, particularly for specialties represented by very few encounters.

The frequency distribution is therefore examined before determining whether less frequent specialties should be consolidated into a broader category.

### 4.6.1 Inspect Medical Specialty Distribution

The distribution of `medical_specialty` is examined to determine how concentrated the encounters are among common specialties and how many categories occur infrequently.

This assessment will guide the consolidation of rare specialties while preserving commonly observed specialties as distinct categories.

In [11]:
# Examine medical specialty distribution
specialty_summary = (
    df_model["medical_specialty"]
    .value_counts()
    .rename_axis("Medical Specialty")
    .reset_index(name="Count")
)

specialty_summary["Percentage"] = (
    specialty_summary["Count"] / len(df_model) * 100
).round(2)

specialty_summary

,Medical Specialty,Count,Percentage
0,Unknown,49947,49.08
1,InternalMedicine,14635,14.38
2,Emergency/Trauma,7565,7.43
3,Family/GeneralPractice,7440,7.31
4,Cardiology,5351,5.26
...,...,...,...
68,Dermatology,1,0.00
69,SportsMedicine,1,0.00
70,Speech,1,0.00
71,Perinatology,1,0.00


In [12]:
# Examine frequency thresholds
for threshold in [100, 500, 1000]:
    n_specialties = (
        specialty_summary["Count"] < threshold
    ).sum()

    n_encounters = specialty_summary.loc[
        specialty_summary["Count"] < threshold,
        "Count"
    ].sum()

    print(
        f"Specialties with fewer than {threshold:,} encounters: "
        f"{n_specialties} specialties, "
        f"{n_encounters:,} encounters "
        f"({n_encounters / len(df_model) * 100:.2f}%)"
    )

Specialties with fewer than 100 encounters: 44 specialties, 925 encounters (0.91%)
Specialties with fewer than 500 encounters: 56 specialties, 3,510 encounters (3.45%)
Specialties with fewer than 1,000 encounters: 63 specialties, 8,340 encounters (8.20%)


### 4.6.2 Consolidate Rare Medical Specialties

The frequency analysis shows that 56 of the 73 medical specialties contain fewer than 500 encounters, yet collectively represent only 3.45% of the dataset. These low-frequency categories may generate sparse features during categorical encoding.

Medical specialties represented by fewer than 500 encounters are therefore consolidated into a single `Other Specialty` category. More frequently observed specialties are retained individually, preserving most of the original specialty information while reducing feature sparsity.

In [13]:
# Identify specialties with fewer than 500 encounters
specialty_counts = df_model["medical_specialty"].value_counts()

rare_specialties = specialty_counts[
    specialty_counts < 500
].index

# Consolidate rare specialties
df_model["medical_specialty"] = (
    df_model["medical_specialty"]
    .replace(rare_specialties, "Other Specialty")
)

print(
    "Number of medical specialty categories:",
    df_model["medical_specialty"].nunique()
)

df_model["medical_specialty"].value_counts()

Number of medical specialty categories: 18


medical_specialty
Unknown                            49947
InternalMedicine                   14635
Emergency/Trauma                    7565
Family/GeneralPractice              7440
Cardiology                          5351
Other Specialty                     3510
Surgery-General                     3099
Nephrology                          1613
Orthopedics                         1400
Orthopedics-Reconstructive          1233
Radiologist                         1140
Pulmonology                          871
Psychiatry                           854
Urology                              685
ObstetricsandGynecology              671
Surgery-Cardiovascular/Thoracic      652
Gastroenterology                     564
Surgery-Vascular                     533
Name: count, dtype: int64

## 4.7 Medication Feature Assessment

The dataset contains multiple variables describing changes in individual diabetes medications. Although these variables contain more than one category, some medications may have been prescribed in only a very small number of encounters.

Features with extremely low variation can contribute little predictive information while unnecessarily increasing the dimensionality of the encoded feature space. The medication variables are therefore examined to identify highly imbalanced or near-constant features before finalizing the predictor set.

### 4.7.1 Inspect Medication Feature Variation

For each medication variable, the frequency of its most common category is calculated. A very high dominant-category percentage indicates that the feature contains little variation across encounters.

This assessment is used to identify near-constant medication variables that may be candidates for removal before categorical encoding.

In [14]:
medication_cols = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone"
]

medication_variation = []

for col in medication_cols:
    counts = df_model[col].value_counts()
    
    medication_variation.append({
        "Medication": col,
        "Unique Values": df_model[col].nunique(),
        "Most Common Category": counts.index[0],
        "Most Common Count": counts.iloc[0],
        "Dominant Category (%)": round(
            counts.iloc[0] / len(df_model) * 100, 2
        )
    })

medication_variation = (
    pd.DataFrame(medication_variation)
    .sort_values("Dominant Category (%)", ascending=False)
    .reset_index(drop=True)
)

medication_variation

,Medication,Unique Values,Most Common Category,Most Common Count,Dominant Category (%)
0,metformin-pioglitazone,2,No,101762,100.00
1,acetohexamide,2,No,101762,100.00
2,metformin-rosiglitazone,2,No,101761,100.00
3,glimepiride-pioglitazone,2,No,101762,100.00
4,troglitazone,2,No,101760,100.00
5,glipizide-metformin,2,No,101750,99.99
6,tolbutamide,2,No,101740,99.98
7,miglitol,4,No,101725,99.96
8,tolazamide,3,No,101724,99.96
9,chlorpropamide,4,No,101677,99.92


### 4.7.2 Remove Near-Constant Medication Features

The medication assessment shows that several variables are almost entirely represented by a single category. Features where more than 99.9% of encounters belong to the dominant category provide very limited variation and would create highly sparse features during categorical encoding.

These near-constant medication variables are therefore removed, while medications with greater variation are retained for modeling.

In [15]:
near_constant_medications = medication_variation.loc[
    medication_variation["Dominant Category (%)"] > 99.9,
    "Medication"
].tolist()

df_model.drop(columns=near_constant_medications, inplace=True)

print("Removed:", near_constant_medications)
print("Dataset shape:", df_model.shape)

Removed: ['metformin-pioglitazone', 'acetohexamide', 'metformin-rosiglitazone', 'glimepiride-pioglitazone', 'troglitazone', 'glipizide-metformin', 'tolbutamide', 'miglitol', 'tolazamide', 'chlorpropamide']
Dataset shape: (101763, 35)


## 4.8 Administrative Categorical Features

The administrative variables `admission_type_id`, `discharge_disposition_id`, and `admission_source_id` are stored as numerical codes, but they represent categories rather than continuous quantities.

These variables are therefore converted to categorical string values before preprocessing so that they can be encoded appropriately during model preparation.

In [16]:
categorical_ids = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

df_model[categorical_ids] = df_model[categorical_ids].astype(str)

In [17]:
df_model[categorical_ids].dtypes

admission_type_id           str
discharge_disposition_id    str
admission_source_id         str
dtype: object

## 4.9 Train-Test Split

Because some patients have multiple hospital encounters, a patient-aware split is used to prevent encounters from the same patient appearing in both the training and test sets.

This helps reduce data leakage and provides a more realistic estimate of model performance on unseen patients.

In [19]:
X = df_model.drop(columns="readmitted_30")
y = df_model["readmitted_30"]
groups = df_model["patient_nbr"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

In [20]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print(
    "Patient overlap:",
    len(
        set(X_train["patient_nbr"]) &
        set(X_test["patient_nbr"])
    )
)

Train shape: (81670, 34)
Test shape: (20093, 34)
Patient overlap: 0


### 4.9.1 Verify Target Distribution

The proportion of 30-day readmissions is compared across the training and test sets to confirm that the patient-aware split maintains a reasonably similar target distribution.

In [21]:
print(f"Train readmission rate: {y_train.mean():.2%}")
print(f"Test readmission rate: {y_test.mean():.2%}")

Train readmission rate: 11.22%
Test readmission rate: 10.92%


### 4.9.2 Remove Patient Identifier

After completing the patient-aware split, `patient_nbr` is no longer required. Since it functions as an identifier rather than a meaningful clinical predictor, it is removed from both the training and test feature sets.

In [22]:
X_train.drop(columns="patient_nbr", inplace=True)
X_test.drop(columns="patient_nbr", inplace=True)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (81670, 33)
Test shape: (20093, 33)


## 4.10 Define Numerical and Categorical Features

The remaining predictors are separated into numerical and categorical features so that appropriate preprocessing can be applied to each group.

Numerical features will be standardized, while categorical features will be converted into model-ready representations using one-hot encoding.

In [23]:
numerical_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))

Numerical features: 8
Categorical features: 25


## 4.11 Build the Preprocessing Pipeline

The numerical and categorical predictors require different preprocessing steps before they can be used for model training.

Numerical features are standardized using `StandardScaler` so that variables measured on different scales contribute more comparably to models that are sensitive to feature magnitude.

Categorical features are transformed using one-hot encoding. `handle_unknown="ignore"` is used so that categories appearing in the test set but not observed in the training set do not cause errors during prediction.

These transformations are combined using a `ColumnTransformer`, allowing numerical and categorical variables to be processed appropriately within a single preprocessing workflow.

In [24]:
numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

## 4.12 Preprocessing Summary

The dataset has now been prepared for predictive modeling of 30-day hospital readmission.

Key preprocessing steps included:

- Creation of a binary 30-day readmission target.
- Removal of encounter and patient identifiers from the final predictor set.
- Grouping of high-cardinality diagnosis codes into broader clinical categories.
- Consolidation of rare medical specialties.
- Removal of near-constant medication features.
- Conversion of administrative ID variables to categorical features.
- A patient-aware train-test split to prevent encounters from the same patient appearing in both datasets.
- Separation of the final 33 predictors into 8 numerical and 25 categorical features.
- Construction of a preprocessing pipeline that standardizes numerical variables and one-hot encodes categorical variables.

The final training set contains 81,670 encounters and the test set contains 20,093 encounters, with no patient overlap between the two sets. The 30-day readmission prevalence remains similar across the partitions (11.22% in training and 10.92% in testing).

The preprocessing transformer will be incorporated directly into the model pipelines so that all learned transformations are fitted using training data only.